<a href="https://colab.research.google.com/github/Jefferson-Aggor/Data-visualization-1/blob/main/Week_Dashboard_Building.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install dash
!pip install dash-bootstrap-components
!pip install plotly
!pip install yfinance

In [31]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

In [32]:
app = Dash(__name__)

In [38]:
# ======================================================================
# Data (loaded once at startup)
# ======================================================================
df = (
    pd.read_csv('stocks.csv', parse_dates=['Date'])
    .sort_values(['Ticker', 'Date'])
    .reset_index(drop=True)
)
TICKERS = sorted(df['Ticker'].unique())
MIN_DATE, MAX_DATE = df['Date'].min(), df['Date'].max()

# Non-trading days (weekends + holidays) so the charts have no gaps
_all_days = pd.bdate_range(MIN_DATE, MAX_DATE)
HOLIDAYS = _all_days.difference(df['Date'].unique()).strftime('%Y-%m-%d').tolist()
RANGEBREAKS = [dict(bounds=['sat', 'mon']), dict(values=HOLIDAYS)]

# Indicators computed on the FULL history, so they stay correct when the
# user narrows the date range.
g = df.groupby('Ticker')['Close']
df['MA5'] = g.transform(lambda s: s.rolling(5).mean())
df['MA20'] = g.transform(lambda s: s.rolling(20).mean())
_std20 = g.transform(lambda s: s.rolling(20).std())
df['BB_UP'] = df['MA20'] + 2 * _std20
df['BB_LO'] = df['MA20'] - 2 * _std20

In [34]:
# Theme
TEMPLATE = 'plotly_white'
C = dict(
    bg='#f4f6fa', card='#ffffff', border='#e3e8ef', text='#1f2937',
    muted='#6b7280', up='#16a34a', down='#dc2626',
    ma5='#f59e0b', ma20='#7c3aed', bb='rgba(37,99,235,0.7)',
    bb_fill='rgba(37,99,235,0.07)', line='#2563eb', fill='rgba(37,99,235,0.12)',
)
FONT = 'Inter, -apple-system, Segoe UI, Roboto, sans-serif'

CARD = dict(backgroundColor=C['card'], border=f"1px solid {C['border']}",
            borderRadius='10px', padding='16px',
            boxShadow='0 1px 3px rgba(16,24,40,0.06)')
LABEL = dict(color=C['muted'], fontSize='12px', textTransform='uppercase',
             letterSpacing='0.06em', marginBottom='6px')

In [35]:
# ======================================================================
# Helpers
# ======================================================================
def empty_figure(message):
    fig = go.Figure()
    fig.update_layout(
        template=TEMPLATE, paper_bgcolor=C['card'], plot_bgcolor=C['card'],
        title=dict(text=message, x=0.5), height=400,
        xaxis={'visible': False}, yaxis={'visible': False},
    )
    return fig


def kpi_card(label, value, sub=None, sub_color=None):
    return html.Div(style={**CARD, 'flex': '1', 'minWidth': '170px'}, children=[
        html.Div(label, style=LABEL),
        html.Div(value, style={'fontSize': '24px', 'fontWeight': '600'}),
        html.Div(sub or '\u00a0', style={'fontSize': '13px', 'marginTop': '4px',
                                         'color': sub_color or C['muted']}),
    ])


def control(label, component, width='auto'):
    return html.Div(style={'width': width}, children=[
        html.Div(label, style=LABEL), component])

In [36]:
# ======================================================================
# Layout
# ======================================================================
app = Dash(__name__)
app.title = 'Stock Dashboard'

app.layout = html.Div(
    style={'backgroundColor': C['bg'], 'color': C['text'], 'fontFamily': FONT,
           'minHeight': '100vh', 'padding': '24px 32px'},
    children=[
        # Header
        html.Div(style={'marginBottom': '20px'}, children=[
            html.H1('Stock Market Dashboard',
                    style={'margin': 0, 'fontSize': '26px', 'fontWeight': '600'}),
            html.Div(f"Daily prices · {MIN_DATE:%b %d, %Y} – {MAX_DATE:%b %d, %Y}",
                     style={'color': C['muted'], 'fontSize': '14px', 'marginTop': '4px'}),
        ]),

        # Controls
        html.Div(style={**CARD, 'display': 'flex', 'gap': '28px', 'flexWrap': 'wrap',
                        'alignItems': 'flex-end', 'marginBottom': '16px'}, children=[
            control('Ticker', dcc.Dropdown(
                id='stock-selector',
                options=[{'label': t, 'value': t} for t in TICKERS],
                value=TICKERS[0], clearable=False), width='180px'),
            control('Chart type', dcc.RadioItems(
                id='chart-type',
                options=[{'label': 'Candlestick', 'value': 'candle'},
                         {'label': 'OHLC', 'value': 'ohlc'},
                         {'label': 'Line', 'value': 'line'},
                         {'label': 'Area', 'value': 'area'}],
                value='candle', inline=True,
                inputStyle={'marginRight': '4px', 'marginLeft': '12px'})),
            control('Overlays', dcc.Checklist(
                id='indicators',
                options=[{'label': 'MA 5', 'value': 'ma5'},
                         {'label': 'MA 20', 'value': 'ma20'},
                         {'label': 'Bollinger (20, 2σ)', 'value': 'bb'},
                         {'label': 'Volume', 'value': 'volume'}],
                value=['ma5', 'ma20', 'volume'], inline=True,
                inputStyle={'marginRight': '4px', 'marginLeft': '12px'})),
            control('Date range', dcc.DatePickerRange(
                id='date-range',
                min_date_allowed=MIN_DATE, max_date_allowed=MAX_DATE,
                start_date=MIN_DATE, end_date=MAX_DATE,
                display_format='MMM D, YYYY')),
        ]),

        # KPI row
        html.Div(id='kpi-row', style={'display': 'flex', 'gap': '16px',
                                      'flexWrap': 'wrap', 'marginBottom': '16px'}),

        # Main chart
        html.Div(style={**CARD, 'padding': '8px', 'marginBottom': '16px'}, children=[
            dcc.Graph(id='stock-price-chart', config={'displaylogo': False}),
        ]),

        # Comparison chart
        html.Div(style={**CARD, 'padding': '8px'}, children=[
            dcc.Graph(id='comparison-chart', config={'displaylogo': False}),
        ]),

        html.Div('Data source: local stocks.csv · For informational purposes only.',
                 style={'color': C['muted'], 'fontSize': '12px', 'marginTop': '16px',
                        'textAlign': 'center'}),
    ],
)


In [37]:
# ======================================================================
# Callbacks
# ======================================================================
@app.callback(
    Output('stock-price-chart', 'figure'),
    Output('kpi-row', 'children'),
    Input('stock-selector', 'value'),
    Input('chart-type', 'value'),
    Input('indicators', 'value'),
    Input('date-range', 'start_date'),
    Input('date-range', 'end_date'),
)
def update_chart(selected_stock, chart_type, indicators, start_date, end_date):
    try:
        indicators = indicators or []
        data = df[(df['Ticker'] == selected_stock)
                  & (df['Date'] >= start_date) & (df['Date'] <= end_date)]

        if data.empty:
            return empty_figure(f'No data available for {selected_stock}'), []

        show_volume = 'volume' in indicators
        rows = 2 if show_volume else 1
        fig = make_subplots(
            rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.03,
            row_heights=[0.75, 0.25] if show_volume else [1.0])

        # --- Price trace ---------------------------------------------------
        if chart_type == 'candle':
            price = go.Candlestick(
                x=data['Date'], open=data['Open'], high=data['High'],
                low=data['Low'], close=data['Close'], name='Price',
                increasing_line_color=C['up'], decreasing_line_color=C['down'],
                increasing_fillcolor=C['up'], decreasing_fillcolor=C['down'])
        elif chart_type == 'ohlc':
            price = go.Ohlc(
                x=data['Date'], open=data['Open'], high=data['High'],
                low=data['Low'], close=data['Close'], name='Price',
                increasing_line_color=C['up'], decreasing_line_color=C['down'])
        else:
            price = go.Scatter(
                x=data['Date'], y=data['Close'], name='Close',
                line=dict(color=C['line'], width=2),
                fill='tozeroy' if chart_type == 'area' else None,
                fillcolor=C['fill'])
        fig.add_trace(price, row=1, col=1)

        # --- Overlays ------------------------------------------------------
        if 'bb' in indicators:
            fig.add_trace(go.Scatter(
                x=data['Date'], y=data['BB_UP'], name='BB upper',
                line=dict(color=C['bb'], width=1, dash='dot')), row=1, col=1)
            fig.add_trace(go.Scatter(
                x=data['Date'], y=data['BB_LO'], name='BB lower',
                line=dict(color=C['bb'], width=1, dash='dot'),
                fill='tonexty', fillcolor=C['bb_fill']), row=1, col=1)
        if 'ma5' in indicators:
            fig.add_trace(go.Scatter(
                x=data['Date'], y=data['MA5'], name='MA 5',
                line=dict(color=C['ma5'], width=1.5)), row=1, col=1)
        if 'ma20' in indicators:
            fig.add_trace(go.Scatter(
                x=data['Date'], y=data['MA20'], name='MA 20',
                line=dict(color=C['ma20'], width=1.5)), row=1, col=1)

        # --- Volume --------------------------------------------------------
        if show_volume:
            colors = [C['up'] if c >= o else C['down']
                      for c, o in zip(data['Close'], data['Open'])]
            fig.add_trace(go.Bar(
                x=data['Date'], y=data['Volume'], name='Volume',
                marker_color=colors, opacity=0.55), row=2, col=1)
            fig.update_yaxes(title_text='Volume', row=2, col=1)

        # --- KPIs ----------------------------------------------------------
        first, last = data['Close'].iloc[0], data['Close'].iloc[-1]
        change = last - first
        pct = change / first * 100
        up = change >= 0
        arrow = '▲' if up else '▼'
        chg_color = C['up'] if up else C['down']

        daily_ret = data['Close'].pct_change().dropna()
        volatility = daily_ret.std() * 100 if len(daily_ret) > 1 else float('nan')

        kpis = [
            kpi_card('Last close', f'${last:,.2f}',
                     f'{arrow} {change:+,.2f} ({pct:+.1f}%)', chg_color),
            kpi_card('Period high', f"${data['High'].max():,.2f}",
                     f"on {data.loc[data['High'].idxmax(), 'Date']:%b %d}"),
            kpi_card('Period low', f"${data['Low'].min():,.2f}",
                     f"on {data.loc[data['Low'].idxmin(), 'Date']:%b %d}"),
            kpi_card('Avg volume', f"{data['Volume'].mean() / 1e6:,.1f}M",
                     f'{len(data)} trading days'),
            kpi_card('Daily volatility', f'{volatility:.2f}%', 'std of daily returns'),
        ]

        # --- Layout --------------------------------------------------------
        fig.update_layout(
            template=TEMPLATE,
            paper_bgcolor=C['card'], plot_bgcolor=C['card'],
            font=dict(family=FONT, color=C['text']),
            title=dict(text=f'{selected_stock} · Price Analysis', x=0.02,
                       font=dict(size=18)),
            height=640 if show_volume else 520,
            hovermode='x unified',
            legend=dict(orientation='h', y=1.07, x=1, xanchor='right'),
            margin=dict(l=50, r=20, t=80, b=30),
            xaxis_rangeslider_visible=False,
        )
        fig.update_xaxes(rangebreaks=RANGEBREAKS, gridcolor=C['border'],
                         showspikes=True, spikethickness=1, spikecolor=C['muted'])
        fig.update_yaxes(gridcolor=C['border'])
        fig.update_yaxes(title_text='Price (USD)', tickprefix='$', row=1, col=1)

        return fig, kpis

    except Exception as e:
        return empty_figure(f'Error loading data: {e}'), []


@app.callback(
    Output('comparison-chart', 'figure'),
    Input('date-range', 'start_date'),
    Input('date-range', 'end_date'),
    Input('stock-selector', 'value'),
)
def update_comparison(start_date, end_date, selected_stock):
    """All tickers rebased to 100 at the start of the selected range."""
    try:
        data = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]
        if data.empty:
            return empty_figure('No data in selected range')

        palette = ['#2563eb', '#f59e0b', '#7c3aed', '#16a34a']
        fig = go.Figure()
        for i, t in enumerate(TICKERS):
            d = data[data['Ticker'] == t]
            if d.empty:
                continue
            rebased = d['Close'] / d['Close'].iloc[0] * 100
            fig.add_trace(go.Scatter(
                x=d['Date'], y=rebased, name=t, mode='lines',
                line=dict(color=palette[i % len(palette)],
                          width=3 if t == selected_stock else 1.5),
                opacity=1 if t == selected_stock else 0.6,
                hovertemplate='%{y:.1f}'))
        fig.add_hline(y=100, line_dash='dot', line_color=C['muted'])

        fig.update_layout(
            template=TEMPLATE,
            paper_bgcolor=C['card'], plot_bgcolor=C['card'],
            font=dict(family=FONT, color=C['text']),
            title=dict(text='Relative performance (start = 100)', x=0.02,
                       font=dict(size=18)),
            height=380, hovermode='x unified',
            legend=dict(orientation='h', y=1.12, x=1, xanchor='right'),
            margin=dict(l=50, r=20, t=70, b=30),
        )
        fig.update_xaxes(rangebreaks=RANGEBREAKS, gridcolor=C['border'])
        fig.update_yaxes(gridcolor=C['border'])
        return fig

    except Exception as e:
        return empty_figure(f'Error loading data: {e}')

In [39]:
# ======================================================================
# Run
# ======================================================================
app.run(jupyter_mode='inline', height=1400, port=8051)

<IPython.core.display.Javascript object>